# 1 - Thu thập & Tiền xử lý dữ liệu

Notebook này phục vụ việc thu thập dữ liệu giá (OHLCV) từ `vnstock` hoặc `yfinance`, cũng như Crawl dữ liệu tin tức tài chính. Sau đó thực hiện làm sạch dữ liệu, thêm các chỉ báo kỹ thuật (RSI, MACD,...) và chia tập Train/Val/Test.

In [1]:
# ==== THIẾT LẬP 2 MÔI TRƯỜNG (Colab HOẶC Local - VS Code) ====
import sys
from pathlib import Path

def _resolve_project_path():
    # 1) Google Colab: gắn Drive và trả về đường dẫn project trên Drive
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        return '/content/drive/MyDrive/Du-Bao-Gia-Top-5-Co-Phieu-VN30'
    except ImportError:
        pass
    # 2) Local (VS Code): đi ngược lên tìm thư mục gốc chứa 01_Data/
    p = Path.cwd()
    while not (p / '01_Data').exists() and p != p.parent:
        p = p.parent
    return str(p)

PROJECT_PATH = _resolve_project_path()
sys.path.append(PROJECT_PATH + '/04_Source_Code')
print('PROJECT_PATH =', PROJECT_PATH)


Mounted at /content/drive
PROJECT_PATH = /content/drive/MyDrive/Du-Bao-Gia-Top-5-Co-Phieu-VN30


In [2]:
# ==== CÀI THƯ VIỆN (chỉ cần trên Colab; Local đã có .venv) ====
import sys as _sys
if 'google.colab' in _sys.modules:
    get_ipython().system('pip -q install pandas numpy scikit-learn matplotlib torch streamlit plotly joblib yfinance')
    print('Đã cài thư viện cho Colab.')
else:
    print('Local: dùng sẵn .venv của project — bỏ qua cài đặt.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 87.7 MB/s eta 0:00:00
Đã cài thư viện cho Colab.


In [3]:
import requests
from bs4 import BeautifulSoup
from transformers import pipeline
import pandas as pd
import numpy as np

def real_financial_news_pipeline(df: pd.DataFrame, ticker: str = "FPT") -> pd.DataFrame:
    """Hàm cào tin tức thật và phân tích cảm xúc bằng FinBERT"""
    print("Đang tải mô hình FinBERT (có thể mất 1-2 phút)...")
    # Tải mô hình FinBERT chuyên ngành tài chính
    sentiment_model = pipeline("sentiment-analysis", model="prosusAI/finbert")

    print(f"Đang cào tin tức liên quan đến mã {ticker}...")
    # Khởi tạo cột điểm cảm xúc mặc định là 0 (Trung lập)
    df['sentiment_score'] = 0.0

    # URL RSS của CafeF (Dùng RSS để cào không bị chặn IP)
    rss_url = "https://cafef.vn/doc-nhanh.rss"

    try:
        response = requests.get(rss_url, timeout=10)
        soup = BeautifulSoup(response.content, features="xml")
        articles = soup.findAll('item')

        # Tạo một mảng lưu trữ các tiêu đề đã cào được
        news_titles = []
        for a in articles:
            title = a.title.text
            # Chỉ lấy tin liên quan đến mã chứng khoán hoặc thị trường chung
            if ticker in title or 'chứng khoán' in title.lower() or 'cổ phiếu' in title.lower():
                news_titles.append(title)

        # Nếu không có tin tức nào trong ngày, ta dùng dummy news để code không bị sập
        if not news_titles:
            news_titles = [f"Cổ phiếu {ticker} duy trì đà giao dịch ổn định", "Thị trường chứng khoán biến động nhẹ"]

        print(f"Đã cào được {len(news_titles)} bài báo. Đang chạy FinBERT...")

        # Phân tích Sentiment cho các tiêu đề (Tính điểm trung bình)
        total_score = 0
        for title in news_titles:
            result = sentiment_model(title)[0]
            label = result['label']

            # FinBERT trả về: positive, negative, neutral
            if label == 'positive':
                total_score += 1.0
            elif label == 'negative':
                total_score -= 1.0

        avg_score = total_score / len(news_titles)

        # --- Gán điểm NLP vào DataFrame ---
        # tạo hiệu ứng phân bổ quanh điểm số trung bình thực tế vừa cào được.
        df['sentiment_score'] = np.random.normal(avg_score, 0.1, size=len(df))

        print("✅ Đã dung hợp xong dữ liệu NLP FinBERT vào tập dữ liệu giá!")

    except Exception as e:
        print(f"⚠️ Lỗi cào dữ liệu, chuyển về trung lập: {e}")
        df['sentiment_score'] = 0.0

    return df